In [ ]:
from com.example.agentic.agent.LLMManager import LLMManager
# 
llm = LLMManager.create_llm('ollama')
llm.call('Hello')

#### LOAD Documents

In [ ]:
from com.example.agentic.loader.LoadManager import LoadManager
# Website Data Ingestion 
documents = LoadManager.from_url(["https://docs.crewai.com/how-to/Installing-CrewAI/"])

#### Chunkings

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

#
print(len(chunks))


##### Embaddings

In [ ]:
from com.example.agentic.embedding.EmbeddingManager import EmbeddingManager
from com.example.agentic.vectors.VectorStoreManager import VectorStoreManager

#chunks=[doc.page_content for doc in documents]

### Convert the text to embeddings
embedding_manager=EmbeddingManager()
embeddings = embedding_manager.embed_documents(documents)
print("[INFO] Example embedding:", embeddings[0] if len(embeddings) > 0 else None)

##store int the vector dtaabase
vectorstore = VectorStoreManager()
vectorstore.add_documents(documents,embeddings)

In [ ]:
import os
from crewai_tools import SerperDevTool,ScrapeWebsiteTool,TavilySearchTool

# Perform search for provide topic
websearch_tool = SerperDevTool()

#
scrape_tool = ScrapeWebsiteTool()

# 1. Initialize the tool with image search enabled
tavily_tool = TavilySearchTool(
    api_key= os.environ.get("TAVILY_API_KEY"),
    include_images=True,
    max_results=10
)

#### Agent Response Formatters

In [ ]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool
from pydantic import BaseModel, Field
from typing import List, Dict
from datetime import datetime

class ResearchPoint(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    findings: str = Field(description="The key findings or insights about this topic")
    relevance: str = Field(description="Why this finding is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="Webpage information with title and url",
        default_factory=list
    )

class ResearchOutput(BaseModel):
    research_points: List[ResearchPoint] = Field(description="List of research findings")
    summary: str = Field(description="Brief summary of overall findings")

class ResearchImage(BaseModel):
    title: str = Field(description="The title of the image")
    url: str = Field(description="The url of the image")

class ResearchImageOutput(BaseModel):
    topic: str = Field(description="The details about primary topic")    
    images: List[ResearchImage] = Field(description="List of top images on topic")
    

class ExecutiveReportSection(BaseModel):
    section_emoji: str = Field(description="Section emoji (e.g., 🔍, 📊, 🎯)")
    section_title: str = Field(description="Section title")
    section_content: str = Field(description="Main content of the section")
    key_insights: List[str] = Field(description="Key insights from this section")
    recommendations: List[str] = Field(
        default_factory=list,
        description="Optional recommendations based on findings"
    )
    sources: List[Dict[str, str]] = Field(
        description="Sources with title and URL for this section",
        default_factory=list
    )

class ExecutiveReport(BaseModel):
    report_title: str = Field(description="Title of the report")
    generation_date: str = Field(description="Report generation date")
    executive_summary: str = Field(description="A concise executive summary")
    key_findings: List[Dict[str, str]] = Field(
        description="List of key findings with their sources",
        default_factory=list
    )
    report_sections: List[ExecutiveReportSection] = Field(
        description="Detailed report sections"
    )
    next_steps: List[str] = Field(description="Recommended next steps")
    sources: List[Dict[str, str]] = Field(
        description="All sources used in the report",
        default_factory=list
    )
    references : List[ResearchImage] = Field(description="References of top images on topic")

#### Crew Knowledge Base

In [ ]:
#
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource
from crewai.knowledge.knowledge_config import KnowledgeConfig
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage
from com.example.agentic.tools.config import _embedder_config_ollama

# put in agent config knowledge_config=knowledge_config
knowledge_config = KnowledgeConfig(results_limit=10, score_threshold=0.5)

# Create custom storage with specific embedder
custom_storage = KnowledgeStorage(
    embedder=_embedder_config_ollama,
    collection_name="my_custom_knowledge"
)

text_kw_source = TextFileKnowledgeSource(
    file_paths=[
        "designs.txt", 
        "machine_learning.txt",
        "python_intro.txt"
    ]
)
# json_source.storage = custom_storage
#
from crewai.knowledge.source.json_knowledge_source import JSONKnowledgeSource
json_source = JSONKnowledgeSource(
    file_paths=[
        "designs-research.json"
    ]
)
#json_source.storage = custom_storage

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.knowledge.source.base_knowledge_source import BaseKnowledgeSource
import os,requests,json
from datetime import datetime
from typing import Dict, Any
from pydantic import BaseModel, Field
from com.example.agentic.tools.config import _embedder_config_ollama

class ImagesKnowledgeSource(BaseKnowledgeSource):
    """Knowledge source that fetches relavent images url for design topic."""

    doc_path: str = Field(description="Json doc path")
    limit: int = Field(default=10, description="Number of images to fetch")

    def load_content(self) -> Dict[Any, str]:
        """Fetch and format desing images."""
        try:
            #response = requests.get(
            #    f"{self.api_endpoint}?limit={self.limit}"
            #)
            #response.raise_for_status()
            #data = response.json()

            work_dir = os.getenv("WORK_DIR")
            with open(f"{work_dir}/{self.doc_path}", 'r') as file:
                _json = json.load(file)
                
            _images = _json.get('design_reference_images', [])

            formatted_data = self.validate_content(_images)
            return {self.doc_path: formatted_data}
        except Exception as e:
            raise ValueError(f"Failed to fetch space news: {str(e)}")

    def validate_content(self, images: list) -> str:
        """Format articles into readable text."""
        formatted = "Design Reference Images:\n\n"
        for _image in images:
            formatted += f"""
                Title: {_image['title']}
                Url: {_image['url']}
                -------------------"""
        return formatted

    def add(self) -> None:
        """Process and store the articles."""
        content = self.load_content()
        for _, text in content.items():
            chunks = self._chunk_text(text)
            self.chunks.extend(chunks)

        self._save_documents()

    def aadd(self) -> None:
        pass
        

# Create knowledge source
desing_images = ImagesKnowledgeSource(
    doc_path="notebooks/knowledge/designs-research.json",
    limit=10
)

##### Design Crew

In [ ]:
from datetime import datetime
from crewai import Task

from crewai_tools import ScrapeWebsiteTool, SerperDevTool
from com.example.agentic.tools.config import _embedder_config_ollama, _embedder_config_openai

@CrewBase
class DesignDevelopment():
    """Latest AI Design Development Crew"""

    agents_config = "config/agents.yaml"
    tasks_config = "config/tasks.yaml"

    @agent
    def content_researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['content_researcher'],
            verbose=True,
            tools=[
                SerperDevTool(),
                ScrapeWebsiteTool()
                # DesignSearchTool()
            ],
            #knowledge_sources=[text_kw_source],
            embedder=_embedder_config_openai,
            llm=llm
        )
    
    @agent
    def images_extractor(self) -> Agent:
        return Agent(
            config=self.agents_config['images_extractor'],
            knowledge_sources=[desing_images],
            verbose=True,
            tools=[],
            embedder=_embedder_config_openai,
            llm=llm
        )
    
    @agent
    def reporting_analyst(self) -> Agent:
        return Agent(
            config=self.agents_config['reporting_analyst'],
            verbose=True,
            llm=llm
        )
    
    @agent
    def formatter(self) -> Agent:
        return Agent(
            config=self.agents_config['formatter'],
            verbose=True,
            llm=llm
        )

    @task
    def research_task(self) -> Task:
        return Task(
            config=self.tasks_config['research_task'], 
            output_json=ResearchOutput
        )


    @task
    def extract_images_task(self) -> Task:
        return Task(
            config=self.tasks_config['extract_images_task'],
            output_json=ResearchImageOutput,
            #context=[self.research_task()]
        )
    
    
    @task
    def reporting_task(self) -> Task:
        return Task(
            config=self.tasks_config['reporting_task'],
            output_pydantic=ExecutiveReport
        )

    @task
    def formatting_task(self) -> Task:
        return Task(
            config=self.tasks_config['formatting_task']
        )

	
    @crew
    def crew(self) -> Crew:
        """Creates the DesignDevelopment Crew"""
        return Crew(
            agents=self.agents, # Automatically created by the @agent decorator
            tasks=self.tasks, # Automatically created by the @task decorator
            process=Process.sequential,
            verbose=True,
        )

In [ ]:
from datetime import datetime
_exe_date = datetime.now().strftime('%Y-%m-%d')
_exe_id = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f" DesignDevelopment Crew triggered on {_exe_date} with execution id {_exe_id}")

def run():
    """
    Run the crew.
    """
    inputs = {
        'topic': 'Microservice Design',
        'date': _exe_date,
        'run_id': _exe_id
    }
    DesignDevelopment().crew().kickoff(inputs=inputs)

In [ ]:
run()

In [ ]:
from crewai.tools import tool

@tool("Basic Functionality")
def my_simple_tool(question: str) -> str:
    """Tool to enhance the question provided."""
    tool_output = f"enhanced_question {question} with clarifications"
    return tool_output

In [2]:
from crewai import Agent, Task, Crew, Process, LLM, Memory
from pydantic import BaseModel, Field
from crewai.rag.embeddings.factory import build_embedder

_embedder = build_embedder({ "provider": "ollama", "config": { "model_name": "nomic-embed-text", "url": "http://localhost:11434/api/embeddings" }})

memory = Memory(
    llm=LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434"),
    embedder=_embedder
)

# Define a Pydantic model to validate order data
class OrderInputModel(BaseModel):
    order_id: int
    product_name: str
    quantity: int = Field(gt=0, description="Quantity must be greater than zero")

# Define a Pydantic model for the output structure
class OrderAnalysisOutputModel(BaseModel):
    total_orders: int
    top_product: str
    total_quantity: int

# Create an agent to analyze orders
order_analyst = Agent(
    role='Order Analyst',
    goal='Analyze the list of orders and provide a summary report.',
    backstory="You are an expert in processing and analyzing order data.",
    verbose=True,
    #memory=memory.scope("/agent/order_analyst"),
    llm=LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434")
)

# Define the task using the Pydantic models for input and output validation
order_analysis_task = Task(
    description="Analyze the incoming list of orders and provide a report on total orders, top product, and total quantity.",
    expected_output="provide a report on total orders, top product, and total quantity.",
    input_model=OrderInputModel,  # Pydantic model for input validation
    output_model=OrderAnalysisOutputModel,  # Pydantic model for output structure
    agent=order_analyst
)

# Define the crew and process
order_analysis_crew = Crew(
    agents=[order_analyst],
    tasks=[order_analysis_task],
    process=Process.sequential,
    memory=memory,
    llm=LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434")
)

# Sample input data (list of orders)
input_data = [
    {"order_id": 1, "product_name": "Laptop", "quantity": 2},
    {"order_id": 2, "product_name": "Monitor", "quantity": 5},
    {"order_id": 3, "product_name": "Laptop", "quantity": 3}
]

# Kick off the task
result = order_analysis_crew.kickoff(inputs={'input_data': input_data})

# The result will be validated and structured by the output_model
print(result)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Order Analyst                                                                                           │
│                                                                                                                 │
│  Task: Analyze the incoming list of orders and provide a report on total orders, top product, and total         │
│  quantity.                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Order Analyst                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  { nil                                                                                                          │
│    { search_memory                                                                                              │
│      {                                                                                                          │
│        "queries": [                                                                                             │
│          "total orders",                                                                                        │
│          "top product",                                                                                         │
│          "total quantity"                                                                                       │
│        ]                                                                                                        │
│      }                                                                                                          │
│    }                                                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{ nil
  { search_memory
    {
      "queries": [
        "total orders",
        "top product",
        "total quantity"
      ]
    }
  }
}


####
####